In [1]:
import requests
from sqlalchemy import create_engine
import pandas as pd
import os
import numpy as np
import lc_classifier
import joblib
import sys
import os
from alerce.core import Alerce
import pandas as pd
import h5py as h5

In [2]:
from lc_classifier.utils import create_astro_object

In [3]:
PATH_H5 = '/home/fsoto/Documents/LCsSSL/data/alerce_data/final/dataset_full.h5'
h5_data = h5.File(PATH_H5, 'r')

In [4]:
oids_h5 = h5_data['SNID']
oids_h5 = [oid.decode('utf-8') for oid in oids_h5]

In [5]:
URL = "https://raw.githubusercontent.com/alercebroker/usecases/master/alercereaduser_v4.json"
PARAMS = requests.get(URL).json()['params']
DB_PARAMs = {
                'user': PARAMS['user'],
                'password': PARAMS['password'],
                'host': PARAMS['host'],
                'dbname': PARAMS['dbname']
                }

engine = create_engine(
    f"postgresql+psycopg2://{DB_PARAMs['user']}:{DB_PARAMs['password']}@{DB_PARAMs['host']}/{DB_PARAMs['dbname']}"
)

In [6]:
query_objects = """
SELECT * FROM lc_classifier_top
WHERE ranking = 1 
AND class_name = 'Transient'
LIMIT 100000;
"""
objects_df = pd.read_sql(query_objects, engine)
objects_df

,oid,classifier_name,classifier_version,class_name,probability,ranking
0,ZTF22aagyalo,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.514,1
1,ZTF18admigch,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.556,1
2,ZTF23abcdaxp,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.916,1
3,ZTF22abugmqm,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.402,1
4,ZTF22abfvxfg,lc_classifier_top,hierarchical_rf_1.1.0,Transient,1.000,1
...,...,...,...,...,...,...
39406,ZTF21abulztu,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.952,1
39407,ZTF21abvdnpt,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.654,1
39408,ZTF21abuzokw,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.710,1
39409,ZTF21abulxpc,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.846,1


In [8]:
filtered_objects = objects_df[~objects_df['oid'].isin(oids_h5)]

In [9]:
filtered_objects

,oid,classifier_name,classifier_version,class_name,probability,ranking
0,ZTF22aagyalo,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.514,1
1,ZTF18admigch,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.556,1
2,ZTF23abcdaxp,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.916,1
3,ZTF22abugmqm,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.402,1
4,ZTF22abfvxfg,lc_classifier_top,hierarchical_rf_1.1.0,Transient,1.000,1
...,...,...,...,...,...,...
39406,ZTF21abulztu,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.952,1
39407,ZTF21abvdnpt,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.654,1
39408,ZTF21abuzokw,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.710,1
39409,ZTF21abulxpc,lc_classifier_top,hierarchical_rf_1.1.0,Transient,0.846,1


In [10]:
filtered_objects_oids = filtered_objects['oid'].tolist()

In [11]:
query_objects_n_detections = """
SELECT * from object
WHERE oid IN  ({filtered_objects_oids})
AND ndet>10
""".format(filtered_objects_oids=','.join(f"'{oid}'" for oid in filtered_objects_oids[20000:]))
objects_n_detections_df = pd.read_sql(
    query_objects_n_detections.format(filtered_objects_oids=','.join(f"'{oid}'" for oid in filtered_objects_oids)),
    engine
)

In [12]:
objects_n_detections_df

,oid,ndethist,ncovhist,mjdstarthist,mjdendhist,corrected,stellar,ndet,g_r_max,g_r_max_corr,...,meanra,meandec,sigmara,sigmadec,deltajd,firstmjd,lastmjd,step_id_corr,diffpos,reference_change
0,ZTF17aaaaebw,1524,1870,58092.160370,60781.158299,True,False,912,NaN,NaN,...,83.058023,34.100393,0.003024,0.002504,2436.683044,58344.475255,60781.158299,27.5.6,True,True
1,ZTF17aaadars,404,1706,58089.246724,60757.173229,True,False,96,NaN,NaN,...,74.008157,4.688009,0.005172,0.005155,2276.900694,58480.272535,60757.173229,27.5.0,True,False
2,ZTF17aaadzee,1419,2716,58101.118542,60823.372257,True,False,816,NaN,NaN,...,307.698073,44.339799,0.003704,0.002649,2472.009155,58351.363102,60823.372257,27.5.6,False,False
3,ZTF17aaadznq,1310,2436,58101.118542,60775.453877,True,False,756,NaN,NaN,...,314.826967,47.323849,0.004199,0.002847,2475.055150,58300.398727,60775.453877,27.5.4,True,False
4,ZTF17aaadzrb,57,3052,58101.119086,60625.144502,False,False,27,NaN,NaN,...,325.209601,47.151330,0.124988,0.085000,1992.698299,58632.446204,60625.144502,24.5.2a6,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13532,ZTF21abulxpc,31,1693,59446.502465,59462.506933,True,False,31,NaN,NaN,...,118.760429,35.414530,0.000056,0.000051,16.004468,59446.502465,59462.506933,correction_1.0.6,True,False
13533,ZTF21abulztu,76,2620,58069.279942,59471.471319,False,False,76,NaN,NaN,...,114.150568,22.472885,0.000039,0.000041,24.969317,59446.502002,59471.471319,correction_1.0.6,True,False
13534,ZTF21abuzokw,1896,3123,58131.201713,60494.476401,True,False,148,2.9406,2.782639,...,67.167512,45.954976,0.086453,0.060104,1077.002164,59422.475509,60499.477674,24.5.2a6,False,False
13535,ZTF21abvadqn,48,1345,59449.500428,59478.502708,True,False,46,NaN,NaN,...,122.877979,39.534299,0.000041,0.000024,29.002280,59449.500428,59478.502708,correction_1.0.6,True,False


In [13]:
objects_n_detections_df_oids = objects_n_detections_df['oid'].tolist()

In [15]:
ra = 288.994009
dec = 13.976708
tolerance = 20 / 3600  # 1 arcsecond in degrees

query_allwise_by_coords = f"""
SELECT * FROM allwise
WHERE ABS(ra - {ra}) < {tolerance}
    AND ABS(dec - {dec}) < {tolerance};
"""
allwise_by_coords_df = pd.read_sql(query_allwise_by_coords, engine)
allwise_by_coords_df

,oid_catalog,ra,dec,w1mpro,w2mpro,w3mpro,w4mpro,w1sigmpro,w2sigmpro,w3sigmpro,w4sigmpro,j_m_2mass,h_m_2mass,k_m_2mass,j_msig_2mass,h_msig_2mass,k_msig_2mass
0,J191558.58+135836.5,288.994098,13.976821,8.771,8.899,8.749,8.38,0.023,0.02,0.073,0.408,10.87,9.513,8.986,0.021,0.029,0.02


In [16]:
#explore database 
query = """
SELECT table_name FROM information_schema.tables
WHERE table_schema='alerce'
ORDER BY table_name;
"""
tables = pd.read_sql_query(query, engine)
tables.sort_values(by="table_name")

,table_name
0,alembic_version
1,allwise
2,dataquality
3,detection
4,feature
5,feature_version
6,forced_photometry
7,gaia_ztf
8,lc_classifier
9,lc_classifier_beta


In [20]:
def extract_from_db(oids_list, engine, fill=False):
     oids_chunk = [f"'{oid}'" for oid in oids_list]


     # Query for detections
     query_detections = f"""
     SELECT * FROM detection
     WHERE oid in ({','.join(oids_chunk)});
     """
     detections = pd.read_sql_query(query_detections, con=engine)
     #detections_path = os.path.join(chunk_dir, "detections.parquet")
     #detections.to_parquet(detections_path)
     # Query for xmatch
     query_xmatch = f"""
     SELECT oid, oid_catalog, dist FROM xmatch
     WHERE oid in ({','.join(oids_chunk)}) and catid='allwise';
     """
     xmatch = pd.read_sql_query(query_xmatch, con=engine)
     xmatch = xmatch.sort_values("dist").drop_duplicates("oid")
     oid_catalog = [f"'{oid}'" for oid in xmatch["oid_catalog"].values]
     #print(oid_catalog)
     if oid_catalog == []:
          return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
     
     
     # Query for WISE data
     query_wise = f"""
     SELECT oid_catalog, w1mpro, w2mpro, w3mpro, w4mpro FROM allwise
     WHERE oid_catalog in ({','.join(oid_catalog)});
     """
     wise = pd.read_sql_query(query_wise, con=engine).set_index("oid_catalog")
     wise = pd.merge(xmatch, wise, on="oid_catalog", how="outer")
     wise = wise[["oid", "w1mpro", "w2mpro", "w3mpro", "w4mpro"]].set_index("oid")
     null_oids = wise[wise[["w1mpro", "w2mpro", "w3mpro", "w4mpro"]].isnull().any(axis=1)].index.tolist()
     # null oids should also contain the oids that are not in the xmatch table
     # oids_chunk is a list of quoted oids, so we need to remove the quotes for comparison
     oids_unquoted = [oid.strip("'") for oid in oids_list]
     null_oids += [oid for oid in oids_unquoted if (oid not in wise.index and oid not in null_oids)]

     if null_oids and fill:
          query_objects = f"""
          SELECT oid, meanra, meandec, sigmara, sigmadec FROM object
          WHERE oid in ({','.join(f"'{oid}'" for oid in null_oids)});
          """
          objects = pd.read_sql_query(query_objects, con=engine)
          objects = objects.set_index("oid")
          for oid in null_oids:
               ra, dec = objects.loc[oid, ['meanra', 'meandec']]
               sigmara, sigmadec = objects.loc[oid, ['sigmara', 'sigmadec']]
               # Use a larger radius for the crossmatch
               radius = max(sigmara, sigmadec) * 5 * 3600  # Convert to arcseconds
               response = requests.get(
                    f"https://catshtm.alerce.online/crossmatch?catalog=WISE&ra={ra}&dec={dec}&radius={radius}"
               )
               if response.status_code != 200:
                    w1 = w2 = w3 = w4 = np.nan
               else:
                    data = response.json()
                    w1 = data.get('Mag_W1', {}).get('value', np.nan)
                    w2 = data.get('Mag_W2', {}).get('value', np.nan)
                    w3 = data.get('Mag_W3', {}).get('value', np.nan)
                    w4 = data.get('Mag_W4', {}).get('value', np.nan)
               wise.loc[oid, ['w1mpro', 'w2mpro', 'w3mpro', 'w4mpro']] = [w1, w2, w3, w4]
     wise_ = wise
     # Query for forced photometry
     query_forced_photometry = f"""
     SELECT * FROM forced_photometry
     WHERE oid in ({','.join(oids_chunk)});
     """
     query_reference = f"""
     SELECT oid, rfid, sharpnr, chinr FROM reference
     WHERE oid in ({','.join(oids_chunk)});
     """
     reference = pd.read_sql_query(query_reference, con=engine)


     forced_photometry = pd.read_sql_query(query_forced_photometry, con=engine)
     #forced_photometry_path = os.path.join(chunk_dir, "forced_photometry.parquet")
     #forced_photometry.to_parquet(forced_photometry_path)
     # Query for PS1 data
     query_ps = f"""
     SELECT oid, sgscore1, sgmag1, srmag1, simag1, szmag1, distpsnr1 FROM ps1_ztf
     WHERE oid in ({','.join(oids_chunk)});
     """
     ps = pd.read_sql_query(query_ps, con=engine)
     ps = ps.drop_duplicates("oid").set_index("oid")

     # Merge xmatch and PS1 data
     xmatch_ = pd.concat([wise_, ps], axis=1).reset_index()
     #xmatch_path = os.path.join(chunk_dir, "xmatch.parquet")
     #xmatch.to_parquet(xmatch_path)
     return detections, forced_photometry,xmatch_, reference


In [23]:
#list = ['ZTF17aabgogz','ZTF17aaadzqx']
list = objects_n_detections_df_oids[:100]  # Example list of OIDs, replace with your own list
detections, forced_photometry, xmatch_filled, reference = extract_from_db(list, engine, fill=True)

In [ ]:
xmatch_filled

,oid,w1mpro,w2mpro,w3mpro,w4mpro,sgscore1,sgmag1,srmag1,simag1,szmag1,distpsnr1
0,ZTF17aaafwqo,3.943,2.452,2.185,1.575,0.972917,21.1602,16.8392,13.3210,11.5030,0.097409
1,ZTF17aaajsuu,4.083,2.973,2.015,1.090,0.643833,19.6390,18.0101,12.3710,10.4930,0.192698
2,ZTF17aabpioy,6.099,6.119,4.931,3.698,1.000000,15.4697,12.8690,11.4630,10.4390,0.430908
3,ZTF17aaatjzo,8.079,8.053,7.657,7.261,0.500000,17.4748,15.3496,13.4008,12.4770,0.240151
4,ZTF17aaarzwv,5.137,4.156,2.745,2.043,0.951250,20.3149,16.1765,13.3899,12.4940,0.180554
...,...,...,...,...,...,...,...,...,...,...,...
195,ZTF18aaagazc,16.643,15.611,12.634,8.792,0.815536,19.7956,19.9337,19.6518,19.4608,0.208935
196,ZTF18aaagwgy,11.200,11.248,11.175,8.921,0.708693,18.0787,17.5687,17.7163,14.1778,0.183282
197,ZTF18aaahyvk,NaN,NaN,NaN,NaN,0.500000,15.2040,14.8170,14.7110,17.0187,1.299957
198,ZTF18aaakeko,NaN,NaN,NaN,NaN,0.500000,18.6784,18.0012,17.4527,17.2712,0.679940


In [230]:
#count null values in the xmatch dataframe
print(xmatch.isnull().sum())
print(xmatch_filled.isnull().sum())


oid            0
w1mpro       155
w2mpro       155
w3mpro       155
w4mpro       155
sgscore1       0
sgmag1         0
srmag1         0
simag1         0
szmag1         0
distpsnr1      0
dtype: int64
oid           0
w1mpro       17
w2mpro       17
w3mpro       17
w4mpro       17
sgscore1      0
sgmag1        0
srmag1        0
simag1        0
szmag1        0
distpsnr1     0
dtype: int64


In [183]:
ra = 88.5746268501346
dec = 18.066946529900875
radius = 20
catalog = 'WISE'
url = f"https://catshtm.alerce.online/crossmatch?catalog={catalog}&ra={ra}&dec={dec}&radius={radius}"

response = requests.get(url)

w1 = response.json()['Mag_W1']['value']
w2 = response.json()['Mag_W2']['value']
w3 = response.json()['Mag_W3']['value']
w4 = response.json()['Mag_W4']['value']

In [184]:
response.json()

{'Dec': {'unit': 'deg', 'value': 18.064579},
 'Dist_2MASS': {'unit': 'arcsec', 'value': 0.195},
 'ErrDec': {'unit': 'arcse', 'value': 0.2033},
 'ErrRA': {'unit': 'arcsec', 'value': 0.1934},
 'MagErr_H': {'unit': 'mag', 'value': 0.048},
 'MagErr_J': {'unit': 'mag', 'value': 0.049},
 'MagErr_K': {'unit': 'mag', 'value': 0.078},
 'MagErr_W1': {'unit': 'mag', 'value': 0.037},
 'MagErr_W2': {'unit': 'mag', 'value': 0.113},
 'MagErr_W3': {'unit': 'mag', 'value': None},
 'MagErr_W4': {'unit': 'mag', 'value': None},
 'Mag_H': {'unit': 'mag', 'value': 14.895},
 'Mag_J': {'unit': 'mag', 'value': 15.515},
 'Mag_K': {'unit': 'mag', 'value': 14.555},
 'Mag_W1': {'unit': 'mag', 'value': 14.588},
 'Mag_W2': {'unit': 'mag', 'value': 14.914},
 'Mag_W3': {'unit': 'mag', 'value': 12.465},
 'Mag_W4': {'unit': 'mag', 'value': 8.862},
 'MeanMJD_W1': {'unit': 'MJD', 'value': 55269.0385249},
 'PA_2MASS': {'unit': 'deg', 'value': -167.5},
 'RA': {'unit': 'deg', 'value': 88.5734244},
 'RChi2_W1': {'unit': ' ', 

In [ ]:

print(f"W1: {w1}, W2: {w2}, W3: {w3}, W4: {w4}")

W1: 12.701, W2: 12.753, W3: 12.356, W4: 9.22


In [33]:
xmatch

,oid,w1mpro,w2mpro,w3mpro,w4mpro,sgscore1,sgmag1,srmag1,simag1,szmag1,distpsnr1
0,ZTF18aabfljw,NaN,NaN,NaN,NaN,0.998750,16.6512,15.7122,15.2100,15.0201,0.327712
1,ZTF18aabfaxr,NaN,NaN,NaN,NaN,0.694432,17.1781,17.5652,17.0298,16.2075,0.066909
2,ZTF17aaafwqo,NaN,NaN,NaN,NaN,0.972917,21.1602,16.8392,13.3210,11.5030,0.097409
3,ZTF18aaakvqu,NaN,NaN,NaN,NaN,1.000000,16.3960,13.1700,10.8660,9.3990,0.310307
4,ZTF17aaajsuu,NaN,NaN,NaN,NaN,0.643833,19.6390,18.0101,12.3710,10.4930,0.192698
...,...,...,...,...,...,...,...,...,...,...,...
595,ZTF18aauymtv,NaN,NaN,NaN,NaN,0.503896,21.8992,21.7702,21.5330,21.4829,0.241664
596,ZTF18aauzldt,NaN,NaN,NaN,NaN,0.074860,20.4766,19.9422,19.2260,19.1504,1.714894
597,ZTF18aavabfl,NaN,NaN,NaN,NaN,0.033798,19.3985,18.8871,18.7334,18.7478,1.681410
598,ZTF18aavesgh,NaN,NaN,NaN,NaN,0.980625,20.5601,20.2980,20.2687,19.9730,0.342395


In [129]:
xmatch = xmatch.iloc[0]

In [130]:
reference

,oid,rfid,sharpnr,chinr
0,ZTF17aaadzqx,769120217,-0.009,0.538
1,ZTF17aaadzqx,769120117,-0.050,0.764


In [131]:
first_ao = create_astro_object(
    data_origin='database',
    detections=detections,
    forced_photometry=forced_photometry,
    xmatch=xmatch,
    reference=reference,
)

In [23]:
first_ao.detections

,oid,candid,pid,ra,dec,mjd,brightness,e_brightness,fid,isdiffpos,forced,distnr,rfid,rb,procstatus,tid,sid,unit
aid,,,,,,,,,,,,,,,,,,
aid_ZTF17aaaocpq,ZTF17aaaocpq,591492212615010003,591492212615,88.723121,46.439352,58345.492211,14.369723,0.010484,r,-1,False,0.576634,745120226.0,0.340000,NaN,ZTF,ZTF,magnitude
aid_ZTF17aaaocpq,ZTF17aaaocpq,618509662615015069,618509662615,88.723431,46.439285,58372.509664,14.175764,0.009971,r,1,False,0.424608,745120226.0,0.496667,NaN,ZTF,ZTF,magnitude
aid_ZTF17aaaocpq,ZTF17aaaocpq,624516932615015065,624516932615,88.723467,46.439313,58378.516933,14.156150,0.010894,r,1,False,0.405926,745120226.0,0.566667,NaN,ZTF,ZTF,magnitude
aid_ZTF17aaaocpq,ZTF17aaaocpq,628506792615015080,628506792615,88.723533,46.439276,58382.506794,14.181611,0.009221,r,1,False,0.609815,745120226.0,0.483333,NaN,ZTF,ZTF,magnitude
aid_ZTF17aaaocpq,ZTF17aaaocpq,668383702615010002,668383702615,88.723294,46.439423,58422.383704,14.405385,0.007370,r,-1,False,0.176922,745120226.0,0.643333,NaN,ZTF,ZTF,magnitude
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
aid_ZTF17aaaocpq,ZTF17aaaocpq,3022212432615015009,3022212432615,88.723350,46.439375,60776.212431,3202.774780,113.416674,r,1,False,0.055640,745120226.0,0.734286,NaN,ZTF,ZTF,diff_flux
aid_ZTF17aaaocpq,ZTF17aaaocpq,3022266252615015004,3022266252615,88.723406,46.439372,60776.266250,1817.771123,58.670143,g,1,False,0.098564,745120126.0,0.880000,NaN,ZTF,ZTF,diff_flux
aid_ZTF17aaaocpq,ZTF17aaaocpq,3027245772615015007,3027245772615,88.723305,46.439322,60781.245775,3158.967807,77.277088,r,1,False,0.256934,745120226.0,0.731429,NaN,ZTF,ZTF,diff_flux


In [24]:
first_ao.metadata

,name,value
0,aid,aid_ZTF17aaaocpq
1,oid,ZTF17aaaocpq
2,W1,8.356
3,W2,7.336
4,W3,4.39
5,W4,1.756
6,sgscore1,0.436762
7,sgmag1,15.1409
8,srmag1,13.305
9,distpsnr1,0.467723


In [25]:
from lc_classifier.features.extractors.allwise_colors_feature_extractor import AllwiseColorsFeatureExtractor
extractor = AllwiseColorsFeatureExtractor(bands=['r', 'g'])

extractor.compute_features_single_object(first_ao)

aaaaa


In [26]:
first_ao.features

,name,value,fid,sid,version
0,W1-W2,1.020000,None,ZTF,1.0.0
1,W2-W3,2.946000,None,ZTF,1.0.0
2,W3-W4,2.634000,None,ZTF,1.0.0
3,r-W1,5.644953,None,ZTF,1.0.0
4,g-W1,6.338750,None,ZTF,1.0.0
5,r-W2,6.664953,None,ZTF,1.0.0
6,g-W2,7.358750,None,ZTF,1.0.0
7,r-W3,9.610953,None,ZTF,1.0.0
8,g-W3,10.304750,None,ZTF,1.0.0
9,r-W4,12.244953,None,ZTF,1.0.0


In [11]:
import pandas as pd
import os
folder_path = '/home/fsoto/Documents/LCsSSL/data/astro_objects_batches'
files = os.listdir(folder_path)
astro_objects = []
for filename in files:
    file = pd.read_pickle(os.path.join(folder_path, filename))
    astro_objects += file






: 

In [ ]:
len(astro_objects)